In [1]:
# 配置模型路径与选项（根据需要修改）
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel
import torch

base_model_path = "/root/autodl-tmp/models/Qwen/Qwen3-8B-Base"
adapter_path = "output_customer_service/lora"
local_files_only = True
trust_remote_code = True

/root/miniconda3/envs/le/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 模型一次性加载的 cell：在内核中保存为全局变量，避免重复加载。
if 'PIPELINE' not in globals():
    print('Start loading tokenizer and model (this may take a while) ...')
    tokenizer = AutoTokenizer.from_pretrained(
        base_model_path, trust_remote_code=trust_remote_code, local_files_only=local_files_only
    )

    import os
    # 强制要求 GPU
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA 未找到，无法将模型加载到 GPU 上。请在有 GPU 的机器上运行，或移除此强制限制。')

    # 强制在 GPU 上使用 float16 以最大化显存利用
    torch_dtype = torch.float16

    # 检查 adapter 是否存在
    adapter_config_file = os.path.join(adapter_path, "adapter_config.json")
    if not (os.path.isdir(adapter_path) and os.path.exists(adapter_config_file)):
        raise FileNotFoundError(f"PEFT adapter 不存在或缺少 adapter_config.json：{adapter_path}\n请确认训练输出包含 adapter_config.json 与 adapter 权重。")

    try:
        # 直接把所有权重加载到 GPU（可能会 OOM，如果 OOM 请参考下方建议）
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_path,
            device_map="cuda",
            torch_dtype=torch_dtype,
            trust_remote_code=trust_remote_code,
            local_files_only=local_files_only,
        )

        model = PeftModel.from_pretrained(
            base_model,
            adapter_path,
            device_map="cuda",
        )

        # 再次确保模型移到 GPU 并设置为半精度与 eval
        try:
            model.to('cuda')
        except Exception:
            pass
        model.eval()
        try:
            model.half()
        except Exception:
            pass

        # 打开 use_cache 加速自回归
        try:
            model.config.use_cache = True
        except Exception:
            pass

        # pipeline 指定 device=0 使用第一块 GPU
        PIPELINE = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)
        print('Loaded PEFT model and adapter fully on GPU as PIPELINE')

    except RuntimeError as e:
        # 捕获 OOM 等错误并给出建议
        print('加载到 GPU 时发生错误：', e)
        print('\n建议：')
        print('- 检查 GPU 剩余显存：`nvidia-smi`')
        print('- 若显存不足，考虑使用 offload（已实现到 notebook 的另一个版本），或使用多 GPU/device_map="auto"，或使用量化/更小的模型')
        raise
else:
    print('Model already loaded in this kernel. Reusing PIPELINE')

Model already loaded in this kernel. Reusing PIPELINE


In [49]:
# 推理封装：可以多次运行这个 cell，而不会重新加载模型（除非内核重启）
def generate(prompt, max_new_tokens=512, **kwargs):
    if 'PIPELINE' not in globals():
        raise RuntimeError('PIPELINE not found. Please run the model-loading cell first.')
    with torch.inference_mode():
        gen_kwargs = dict(
            max_new_tokens=max_new_tokens,
            return_full_text=False,
            use_cache=True,
            clean_up_tokenization_spaces=False,
            do_sample=True,        # 关闭采样（更确定）
            temperature=0.8,       # 若 do_sample=True，设低温
            top_p=0.90,
            repetition_penalty=1.1,
        )
        try:
            gen_kwargs.setdefault('pad_token_id', tokenizer.eos_token_id)
        except Exception:
            pass
        gen_kwargs.update(kwargs)

        response = PIPELINE(
            prompt, 
            **gen_kwargs)
        
        gen_text = ''
        if isinstance(response, list) and len(response) > 0:
            item = response[0]
            if isinstance(item, dict):
                gen_text = item.get('generated_text') or item.get('text') or ''
            else:
                gen_text = str(item)
        elif isinstance(response, dict):
            gen_text = response.get('generated_text') or response.get('text') or ''
        else:
            gen_text = str(response)

        if isinstance(prompt, str) and gen_text.startswith(prompt):
            gen_text = gen_text[len(prompt):].lstrip()

        return gen_text

# 示例调用
try:
    messages = [{"role":"system","content":"回答不要出现任何其他语言"},
                {"role":"user","content":"退货政策是什么？购买了八天了"}]
    try:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    except Exception:
        prompt = '退货政策是什么？购买了八天了'
    print(generate(prompt=prompt, max_new_tokens=256))
except Exception as e:
    print('调用生成失败：', e)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


亲，这个商品支持七天无理由退换货哦。您可以在订单页面申请售后，我们收到退回的商品后会原路退款给您。𖠚ᐝ



In [34]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("/root/autodl-tmp/models/Qwen/Qwen3-8B-Base", trust_remote_code=True, local_files_only=True)

messages = [{"role":"system","content":"只用中文回答，不要出现任何其他语言。回答简洁明了。"},
            {"role":"user","content":"退货政策是什么？购买了八天了"}]
# messages = "退货政策是什么？购买了八天了"
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
print("PROMPT:\n", prompt)

PROMPT:
 <|im_start|>system
只用中文回答，不要出现任何其他语言。回答简洁明了。<|im_end|>
<|im_start|>user
退货政策是什么？购买了八天了<|im_end|>
<|im_start|>assistant

